In [2]:
# ============================================================
# FX-AlphaLab: From Row Data To Explainable Alpha
# Correlation Agent Project - Version Complète & Structurée
# ============================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import xgboost as xgb
import shap

# ========================= CONFIG =========================
OUT_DIR = "./FX_AlphaLab_Final"
os.makedirs(OUT_DIR, exist_ok=True)

SPLIT_DATE = "2024-01-01"
PROBA_BUY = 0.60
PROBA_SELL = 0.40
MIN_EXPECTED_LOGRET = 0.0005
Z_THRESHOLD = 4.0   # Seuil pour détecter les outliers

np.random.seed(42)

print("="*100)
print("FX-ALPHALAB - CORRELATION AGENT")
print("Version Complète & Structurée")
print("="*100)

# ============================================================
# 1. BUSINESS UNDERSTANDING & ANALYTIC APPROACH
# ============================================================
print("\n1. BUSINESS UNDERSTANDING & ANALYTIC APPROACH")
print("="*80)

print("""
BUSINESS OBJECTIVES (BO)
BO1 → Enhance estimation performance of trading decisions
BO2 → React quickly and effectively to market events
BO3 → Achieve Superior Signal Quality and Conviction Scoring
BO4 → Reduce portfolio risk and capture hidden profit opportunities

DATA SCIENCE OBJECTIVES (DSO)
DSO1 → Predict performance of trading decisions
DSO2 → Real-time analysis models
DSO3 → Multi-agent signal aggregation and conviction scoring
DSO4 → Dynamic Correlation Intelligence & Arbitrage Detection   ← Objectif principal
""")

# ============================================================
# 2. DATA ACQUISITION
# ============================================================
print("\n2. DATA ACQUISITION")
print("="*80)

MAIN_PAIRS = ["EURUSD", "GBPUSD", "USDJPY", "EURJPY", "USDCHF"]

prices = {}
for pair in MAIN_PAIRS:
    filename = f"{pair}_from_2018.csv"
    if os.path.exists(filename):
        df = pd.read_csv(filename, parse_dates=[0], index_col=0)
        prices[pair] = df['Close'].astype(float)
        print(f"✓ {pair} chargé → {len(df)} lignes")
    else:
        print(f"✗ {filename} non trouvé")

closes = pd.DataFrame(prices).sort_index()
print(f"\nDonnées brutes chargées : {closes.shape[0]} jours × {closes.shape[1]} paires")

# ============================================================
# 3. DATA CLEANING & UNDERSTANDING
# ============================================================
print("\n3. DATA CLEANING & UNDERSTANDING")
print("="*80)

# Avant nettoyage
missing_before = closes.isna().sum().sum()
print(f"Valeurs manquantes avant nettoyage : {missing_before}")

# Détection des outliers (Z-score > 4)
returns_raw = np.log(closes / closes.shift(1))
z_scores = np.abs((returns_raw - returns_raw.mean()) / returns_raw.std())
outliers_before = (z_scores > Z_THRESHOLD).sum().sum()
print(f"Valeurs aberrantes (outliers) avant nettoyage : {outliers_before}")

# Nettoyage
closes_clean = closes.copy()

# Remplir les valeurs manquantes par interpolation
closes_clean = closes_clean.interpolate(method='linear', limit_direction='both')

# Traitement des outliers (capping)
for col in closes_clean.columns:
    mean = closes_clean[col].mean()
    std = closes_clean[col].std()
    lower = mean - Z_THRESHOLD * std
    upper = mean + Z_THRESHOLD * std
    closes_clean[col] = closes_clean[col].clip(lower=lower, upper=upper)

# Après nettoyage
missing_after = closes_clean.isna().sum().sum()
print(f"Valeurs manquantes APRÈS nettoyage : {missing_after} ✓")

# Recalcul des returns après nettoyage
returns = np.log(closes_clean / closes_clean.shift(1)).dropna()
z_scores_after = np.abs((returns - returns.mean()) / returns.std())
outliers_after = (z_scores_after > Z_THRESHOLD).sum().sum()
print(f"Valeurs aberrantes APRÈS nettoyage : {outliers_after} ✓")

closes_aligned = closes_clean.copy()

print(f"\nDonnées finales après nettoyage : {closes_aligned.shape[0]} jours × {closes_aligned.shape[1]} paires")
print(f"Période : {closes_aligned.index.min().date()} → {closes_aligned.index.max().date()}")

# ============================================================
# 4. MODELING (avec Correlation Agent)
# ============================================================
print("\n4. MODELING - XGBoost + Correlation Agent Features")
print("="*80)

def build_features_for_pair(pair: str, closes: pd.DataFrame, returns: pd.DataFrame):
    df = pd.DataFrame(index=returns.index)
    r = returns[pair]
    px = closes[pair]

    # Features classiques
    for lag in [1, 2, 3, 5, 10]:
        df[f"ret_lag_{lag}"] = r.shift(lag)
    for w in [5, 10, 20, 50]:
        df[f"vol_{w}"] = r.rolling(w).std()
        df[f"ret_mean_{w}"] = r.rolling(w).mean()

    # RSI
    delta = px.diff()
    up = delta.clip(lower=0).rolling(14).mean()
    down = (-delta).clip(lower=0).rolling(14).mean()
    rs = up / (down + 1e-12)
    df["rsi_14"] = 100 - (100 / (1 + rs))

    # === Correlation Agent Features ===
    for other in returns.columns:
        if other != pair:
            df[f"corr_{pair}_{other}_20"] = r.rolling(20).corr(returns[other])

    corr_cols = [col for col in df.columns if col.startswith("corr_")]
    df["avg_correlation"] = df[corr_cols].mean(axis=1)
    df["std_correlation"] = df[corr_cols].std(axis=1)
    df["correlation_stability"] = df["avg_correlation"] / (df["std_correlation"] + 1e-6)

    return df

def make_targets(pair: str, closes: pd.DataFrame):
    y_reg = closes[pair].shift(-1)
    y_cls = (closes[pair].shift(-1) > closes[pair]).astype(int)
    return y_reg, y_cls

# ====================== MODELING ======================
results = []

for pair in closes_aligned.columns:
    print(f"\n→ Modeling {pair}")

    X = build_features_for_pair(pair, closes_aligned, returns)
    y_reg, y_cls = make_targets(pair, closes_aligned)

    data = X.copy()
    data["y_reg"] = y_reg
    data["y_cls"] = y_cls
    data = data.dropna()

    train = data[data.index < SPLIT_DATE]
    test = data[data.index >= SPLIT_DATE]

    if len(train) < 300 or len(test) < 100:
        print(f"   ⚠️ Données insuffisantes pour {pair}")
        continue

    X_train = train.drop(["y_reg", "y_cls"], axis=1)
    X_test = test.drop(["y_reg", "y_cls"], axis=1)

    yreg_train = train["y_reg"]
    yreg_test = test["y_reg"]
    ycls_train = train["y_cls"]
    ycls_test = test["y_cls"]

    # Modèles XGBoost
    reg_model = xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                                 subsample=0.8, colsample_bytree=0.8, random_state=42)
    reg_model.fit(X_train, yreg_train)
    yreg_pred = reg_model.predict(X_test)

    cls_model = xgb.XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                                  subsample=0.8, colsample_bytree=0.8, random_state=42)
    cls_model.fit(X_train, ycls_train)
    ycls_pred = cls_model.predict(X_test)
    proba_up = cls_model.predict_proba(X_test)[:, 1]

    # Signaux
    close_today = closes_aligned.loc[X_test.index, pair].values
    expected_logret = np.log((yreg_pred + 1e-12) / (close_today + 1e-12))

    signal = np.where(
        (proba_up >= PROBA_BUY) & (expected_logret >= MIN_EXPECTED_LOGRET), "BUY",
        np.where((proba_up <= PROBA_SELL) & (expected_logret <= -MIN_EXPECTED_LOGRET), "SELL", "HOLD")
    )

    # Backtest
    realized_ret = np.log(closes_aligned[pair].shift(-1).loc[X_test.index] / close_today)
    pnl = np.where(signal == "BUY", realized_ret, np.where(signal == "SELL", -realized_ret, 0))

    results.append({
        "Pair": pair,
        "Cls_Accuracy": float(accuracy_score(ycls_test, ycls_pred)),
        "Sharpe": float(np.mean(pnl) / np.std(pnl) * np.sqrt(252)) if np.std(pnl) > 0 else 0,
        "Total_PnL": float(np.sum(pnl))
    })

summary_df = pd.DataFrame(results)
summary_df.to_csv(os.path.join(OUT_DIR, "model_summary.csv"), index=False)

print("\n" + "="*100)
print("RÉSULTATS FINAUX")
print("="*100)
print(summary_df.round(4))

print(f"\n✅ Travail terminé et bien structuré !")
print(f"Dossier de sortie : {OUT_DIR}")

FX-ALPHALAB - CORRELATION AGENT
Version Complète & Structurée

1. BUSINESS UNDERSTANDING & ANALYTIC APPROACH

BUSINESS OBJECTIVES (BO)
BO1 → Enhance estimation performance of trading decisions
BO2 → React quickly and effectively to market events  
BO3 → Achieve Superior Signal Quality and Conviction Scoring
BO4 → Reduce portfolio risk and capture hidden profit opportunities

DATA SCIENCE OBJECTIVES (DSO)
DSO1 → Predict performance of trading decisions
DSO2 → Real-time analysis models
DSO3 → Multi-agent signal aggregation and conviction scoring
DSO4 → Dynamic Correlation Intelligence & Arbitrage Detection   ← Objectif principal


2. DATA ACQUISITION
✓ EURUSD chargé → 2121 lignes
✓ GBPUSD chargé → 2121 lignes
✓ USDJPY chargé → 2121 lignes
✓ EURJPY chargé → 2121 lignes
✓ USDCHF chargé → 2121 lignes

Données brutes chargées : 2121 jours × 5 paires

3. DATA CLEANING & UNDERSTANDING
Valeurs manquantes avant nettoyage : 0
Valeurs aberrantes (outliers) avant nettoyage : 40
Valeurs manquantes A

In [ ]:
from google.colab import drive
drive.mount('/content/drive')